# Tokenization Parity (TP)

> **Paper:** *Lost in Transliteration: Orthographic Sensitivity in Neural MT Evaluation*  
> **Authors:** G L John Salvin · Swapnil Hingmire · IIT Palakkad · 2026  
> **Notebook:** `01_tokenization_parity.ipynb`

---

This notebook computes **Tokenization Parity (TP)** — a measure of how much more the XLM-RoBERTa tokenizer fragments Indic-script text compared to English. TP is used as evidence of surface-level script bias in COMET and related neural MT evaluation metrics.

The formula is:

$$TP = \frac{\text{token count}_{\text{Indic}}}{\text{token count}_{\text{English}}}$$

| TP value | Interpretation |
|---|---|
| 1.0 | Same token count as English — no tokenizer bias |
| 1.5 | 50% more tokens than English — moderate fragmentation |
| 2.0 | 100% more tokens than English — high fragmentation |

TP is computed separately for native-script and romanized versions of each translation, and for both the reference and the machine translation output.


## Setup

Install dependencies if needed:

```bash
pip install transformers pandas openpyxl
```

The tokenizer used is `xlm-roberta-base`, which is the backbone of COMET v2.2.6 (Rei et al., 2022). We use it to count how many subword tokens each sentence produces under native script vs. romanized form.

**Data path:** place the five per-language CSV files in `../data/processed/` before running. Tokenized outputs will be written to `../data/processed/tokenization_outputs/`.


In [ ]:
import os
import glob
import pandas as pd
from transformers import XLMRobertaTokenizerFast

DATASET_DIR = "../data/processed"
OUTPUT_DIR  = "../data/processed/tokenization_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# columns we want to tokenize — romanized variants are included
TARGET_COLUMNS = [
    "Source",
    "Reference",
    "Translation",
    "Reference_Transliteration",
    "Translation_Transliteration",
    "Reference_Transliteration_clean",
    "Translation_Transliteration_clean",
    "Reference_Transliteration_romanized",
    "Translation_Transliteration_romanized",
]

LANGUAGE_MAP = {
    "gujarati": "Gujarati",
    "hindi":    "Hindi",
    "tamil":    "Tamil",
    "malayalam":"Malayalam",
    "marathi":  "Marathi",
}

# XLM-R base — same tokenizer used inside COMET v2.2.6
print("Loading tokenizer: xlm-roberta-base")
tokenizer = XLMRobertaTokenizerFast.from_pretrained("xlm-roberta-base")
print("Tokenizer loaded.\n")

csv_files = sorted(glob.glob(os.path.join(DATASET_DIR, "*.csv")))
print(f"Found {len(csv_files)} CSV file(s):\n")
for f in csv_files:
    print(f"  {os.path.basename(f)}")


## Tokenizing Each Language File

For every CSV file (one per language), we tokenize each target column using XLM-R. Two new columns are inserted immediately after the source column:

- `<col>_xlmr_tokens` — the full token sequence, joined with ` | ` for readability
- `<col>_xlmr_token_count` — the integer count used in the TP ratio

Columns that don't exist in a particular file (e.g., romanized columns before the romanization pipeline has been run) are skipped with a warning, so the notebook is safe to run at any stage of the pipeline.

The `resolve_columns` helper does case-insensitive, punctuation-agnostic matching, which handles minor column name variations across the five language files.


In [ ]:
def detect_language(filename):
    for keyword, lang in LANGUAGE_MAP.items():
        if keyword in filename.lower():
            return lang
    return "Unknown"


def resolve_columns(df, targets):
    """Match target names to actual DataFrame columns, ignoring case and punctuation."""
    def norm(s):
        return "".join(ch.lower() for ch in s if ch.isalnum())
    col_map = {norm(c): c for c in df.columns}
    return {t: col_map[norm(t)] for t in targets if norm(t) in col_map}


def tokenize_text(text):
    """Return (token_string, token_count) for a single input text."""
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return "", 0
    tokens = tokenizer.tokenize(str(text))
    return " | ".join(tokens), len(tokens)


summary_rows = []

for csv_path in csv_files:
    base_name = os.path.splitext(os.path.basename(csv_path))[0]
    language  = detect_language(base_name)

    print(f"{'='*65}")
    print(f"Language : {language}")
    print(f"File     : {base_name}")
    print(f"{'='*65}")

    df = pd.read_csv(csv_path)
    print(f"Columns in file ({len(df.columns)}):")
    for c in df.columns:
        print(f"  • {c}")
    print()

    resolved = resolve_columns(df, TARGET_COLUMNS)
    missing  = set(TARGET_COLUMNS) - set(resolved.keys())
    if missing:
        print("  Columns not found in this file (skipped):")
        for m in sorted(missing):
            print(f"    - {m}")
        print()

    if not resolved:
        print("  No target columns found — skipping file.\n")
        continue

    # collect insertions first so index shifts don't affect later inserts
    cols_to_insert = []

    print(f"  {'Column':<45} {'Avg':>6} {'Min':>5} {'Max':>5}")
    print(f"  {'-'*45} {'-'*6} {'-'*5} {'-'*5}")

    for requested, actual in resolved.items():
        token_data    = df[actual].apply(tokenize_text)
        tokens_series = token_data.apply(lambda x: x[0])
        counts_series = token_data.apply(lambda x: x[1])

        insert_at = df.columns.get_loc(actual) + 1
        cols_to_insert.append((insert_at, f"{actual}_xlmr_tokens",      tokens_series))
        cols_to_insert.append((insert_at, f"{actual}_xlmr_token_count", counts_series))

        avg, mn, mx = counts_series.mean(), counts_series.min(), counts_series.max()
        print(f"  ✓ {actual:<45} {avg:>6.1f} {mn:>5} {mx:>5}")
        summary_rows.append({"Language": language, "Column": actual,
                              "Avg Tokens": round(avg, 1), "Min Tokens": mn, "Max Tokens": mx})

    for insert_at, col_name, series in sorted(cols_to_insert, key=lambda x: x[0], reverse=True):
        df.insert(insert_at, col_name, series)

    out_path = os.path.join(OUTPUT_DIR, f"{base_name}_xlmr_tokenized.csv")
    df.to_csv(out_path, index=False)
    print(f"\n  → Saved: {out_path}\n")

# cross-language summary table
print(f"{'='*65}")
print("SUMMARY — Average Token Counts Across All Languages")
print(f"{'='*65}")
summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    pivot = summary_df.pivot_table(index="Column", columns="Language",
                                   values="Avg Tokens", aggfunc="first")
    print(pivot.to_string())
    summary_path = os.path.join(OUTPUT_DIR, "tokenization_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSummary saved: {summary_path}")
print("\nAll done.")


## Computing the TP Ratio

Now that every CSV has token counts, we compute TP for each Indic column relative to the English source:

$$TP = \frac{n_{\text{Indic}}}{n_{\text{English}}}$$

where $n$ is the XLM-R subword token count. A TP above 1.0 means XLM-R splits the Indic sentence into more pieces than the English — a direct consequence of how infrequently Indic-script characters appear in the pretraining corpus. This fragmentation inflates the hidden-state representation of Indic text relative to English, which we argue biases downstream metric scores.


In [ ]:
ENGLISH_COUNT_COL = "Source_xlmr_token_count"

TP_PAIRS = [
    ("Reference_xlmr_TP",                         "Reference_xlmr_token_count"),
    ("Translation_xlmr_TP",                       "Translation_xlmr_token_count"),
    ("Reference_Transliteration_romanized_xlmr_TP",
     "Reference_Transliteration_romanized_xlmr_token_count"),
    ("Translation_Transliteration_romanized_xlmr_TP",
     "Translation_Transliteration_romanized_xlmr_token_count"),
]

tp_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*_xlmr_tokenized.csv")))
print(f"Found {len(tp_files)} tokenized file(s)\n")

for path in tp_files:
    fname = os.path.basename(path)
    df    = pd.read_csv(path)

    # guard against zero English token counts to avoid division errors
    english = df[ENGLISH_COUNT_COL].replace(0, float("nan"))

    for tp_col, count_col in TP_PAIRS:
        if count_col not in df.columns:
            print(f"  '{count_col}' not in {fname} — skipped")
            continue
        if tp_col in df.columns:
            continue  # already computed on a previous run
        tp_series = df[count_col] / english
        insert_at = df.columns.get_loc(count_col) + 1
        df.insert(insert_at, tp_col, tp_series)

    df.to_csv(path, index=False)

    lang = next((v for k, v in LANGUAGE_MAP.items() if k in fname.lower()), "Unknown")
    summary = " ".join(
        f"{tp_col.split('_xlmr')[0].split('_Transliteration')[0].replace('_',' ')} "
        f"TP={df[tp_col].mean():.3f}"
        for tp_col, _ in TP_PAIRS if tp_col in df.columns
    )
    print(f"✓ {lang:<12} | {summary}")

print("\nDone — TP columns added and files saved.")
print()
print("TP interpretation:")
print("  1.0 = same token count as English (no tokenizer bias)")
print("  1.5 = 50% more tokens than English (moderate fragmentation)")
print("  2.0 = 100% more tokens than English (high fragmentation)")


## Exporting to Excel

All five tokenized CSV files (one per language, now including token counts and TP ratios) are combined into a single Excel workbook with one sheet per language. This file is the input for the next stage — `02_information_parity.ipynb`.


In [ ]:
excel_path = os.path.join(OUTPUT_DIR, "tokenization_outputs_all.xlsx")
csv_files_out = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.csv")))

if not csv_files_out:
    print(f"No CSV files found in: {OUTPUT_DIR}")
else:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for csv_path in csv_files_out:
            sheet_name = os.path.splitext(os.path.basename(csv_path))[0][:31]
            pd.read_csv(csv_path).to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"✓ Combined Excel saved to: {excel_path}")


---

## References

**Tokenization bias in multilingual models:**
Kanjirangat, V., Samardžić, T., Dolamic, L., & Rinaldi, F. (2025). *Tokenization and Representation Biases in Multilingual Models on Dialectal NLP Tasks.* EMNLP 2025, pp. 23992–24010. https://arxiv.org/abs/2509.20045

**Cross-lingual tokenization fairness:**
Foroutan, N., Meister, C., Paul, D., Niklaus, J., Ahmadi, S., Bosselut, A., & Sennrich, R. (2025). *Parity-Aware Byte-Pair Encoding: Improving Cross-lingual Fairness in Tokenization.* arXiv:2508.04796. https://arxiv.org/abs/2508.04796

**XLM-RoBERTa (tokenizer backbone):**
Conneau, A., Khandelwal, K., Goyal, N., Chaudhary, V., Wenzek, G., Guzmán, F., Grave, E., Ott, M., Zettlemoyer, L., & Stoyanov, V. (2020). *Unsupervised Cross-lingual Representation Learning at Scale.* ACL 2020. https://arxiv.org/abs/1911.02116

**COMET (wmt22-comet-da):**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). *COMET: A Neural Framework for MT Evaluation.* EMNLP 2020. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset:**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., & Dabre, R. (2023). *IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics for Indian Languages.* ACL 2023. https://aclanthology.org/2023.acl-long.795